# DiffuLLaMA smoke test

Standalone — **no `dlmrel` upload needed**. Everything is inline, like the older
notebooks in `~/SAE4DLM-CE/dlm_order/`.

Purpose: prove the whole stack works on this machine before booking a long run.
It checks the three things that fail *silently* (HANDOFF §9):

1. **eager attention** — `sdpa`/`flash_attention_2` accept `output_attentions=True`
   and return `None`, turning every accuracy into zero with no error;
2. **the BOS attention sink** — must be excluded *before* the argmax;
3. **the teacher-forced masking schedule** — that intermediate states rebuild.

Then it reruns a known result as an end-to-end check: on DiffuLLaMA-7B,
**L3 H11** is the relational object→verb head (0.877) and **L18 H10** is the
positional `+1` head (HANDOFF §4.2). If those two don't behave as expected here,
something in the stack is wrong.

Runtime: a few minutes, almost all of it downloading 7B weights.

## 1 · GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), 'No GPU — Runtime > Change runtime type.'
p = torch.cuda.get_device_properties(0)
print(f'\n{p.name} | {p.total_memory / 1e9:.0f} GB')

## 2 · Install and clone

`transformers` pinned to **4.44.2** — later versions changed the attention
plumbing `output_attentions` relies on. DiffuLLaMA is not on PyPI, hence the
clone; `%cd` into it so `from model import ...` resolves.

In [ ]:
!pip install -q transformers==4.44.2 accelerate sentencepiece 'huggingface_hub<0.37'
!rm -rf /content/DiffuLLaMA
!git clone -q --depth 1 https://github.com/HKUNLP/DiffuLLaMA.git /content/DiffuLLaMA
%cd /content/DiffuLLaMA

import transformers
print('transformers', transformers.__version__)
assert transformers.__version__ == '4.44.2', 'wrong transformers — restart and rerun'

## 3 · Load the model

`_attn_implementation='eager'` is the load-bearing argument here.

In [ ]:
import time
from transformers import AutoConfig, AutoTokenizer, LlamaForCausalLM
from model import DiscreteDiffusionModel, get_anneal_attn_mask

MODEL = 'diffusionfamily/diffullama'
DEVICE = 'cuda'

t0 = time.time()
hf_config = AutoConfig.from_pretrained(MODEL)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
backbone = LlamaForCausalLM.from_pretrained(
    MODEL,
    device_map='auto',
    _attn_implementation='eager',   # <- sdpa/flash silently drop attentions
    torch_dtype=torch.bfloat16,
)
model = DiscreteDiffusionModel(
    model=backbone, config=hf_config, tokenizer=tokenizer, device=DEVICE,
).to(DEVICE)
model.eval()

N_LAYERS = hf_config.num_hidden_layers
N_HEADS = hf_config.num_attention_heads
MASK_ID = tokenizer.mask_token_id
BOS_ID = tokenizer.bos_token_id

assert MASK_ID is not None, 'no mask token — the diffusion schedule needs one'
assert tokenizer.is_fast, (
    'slow tokenizer — return_offsets_mapping needs a fast one. '
    'Try AutoTokenizer.from_pretrained(MODEL, use_fast=True).'
)
print(f'\nloaded in {time.time() - t0:.0f}s')
print(f'{N_LAYERS} layers x {N_HEADS} heads = {N_LAYERS * N_HEADS} heads')
print(f'mask_token_id={MASK_ID}  bos_token_id={BOS_ID}')
print(f'GPU allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 4 · Helpers

In [ ]:
import numpy as np

@torch.no_grad()
def forward_with_attentions(input_ids):
    """Returns attentions: tuple of N_LAYERS tensors [1, heads, seq, seq]."""
    embeds = model.get_embeds(input_ids)
    attn_mask = get_anneal_attn_mask(
        seq_len=input_ids.shape[1], bsz=input_ids.shape[0],
        dtype=embeds.dtype, device=input_ids.device, attn_mask_ratio=1.0,
    )
    out = model.denoise_model(
        inputs_embeds=embeds, attention_mask=attn_mask,
        output_attentions=True, output_hidden_states=False,
        return_dict=True, use_cache=False,
    )
    return out.attentions


def encode(text):
    """Token ids with BOS prepended, plus char offsets per token."""
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = [BOS_ID] + enc['input_ids']
    # shift offsets by one to account for the prepended BOS
    offsets = [(-1, -1)] + list(enc['offset_mapping'])
    return torch.tensor([ids], device=DEVICE), offsets


def span_for_word(text, offsets, word):
    """Token indices covering `word` in `text`. Fails loudly if absent."""
    start = text.find(word)
    assert start >= 0, f'{word!r} not in {text!r}'
    end = start + len(word)
    span = [i for i, (s, e) in enumerate(offsets) if s < end and e > start and s >= 0]
    assert span, f'no tokens aligned to {word!r}'
    return span


def receiver_predictions(attentions, layer, attender_span):
    """Argmax column per head for one layer, BOS and self excluded first."""
    row = attentions[layer][0, :, attender_span[-1], :].detach().float().clone()
    row[:, 0] = float('-inf')            # BOS attention sink
    for c in attender_span:
        row[:, c] = float('-inf')        # self
    return row.argmax(dim=1).cpu().numpy()


def teacher_forced_state(true_ids, diffusion_time, steps=64, seed=42):
    """Rebuild x_t. t=0 fully masked; t=steps-1 forced fully visible."""
    torch.manual_seed(seed); np.random.seed(seed)
    maskable = torch.ones_like(true_ids, dtype=torch.bool)
    maskable[:, 0] = False                       # never mask BOS
    xt = true_ids.masked_fill(maskable, MASK_ID)
    remaining = maskable.clone()
    for progress in range(diffusion_time):
        p = 1.0 / (steps - progress)
        reveal = remaining & (torch.rand_like(remaining, dtype=torch.float) < p)
        xt = xt.clone()
        xt[reveal] = true_ids[reveal]
        remaining &= ~reveal
    if diffusion_time == steps - 1:
        xt, remaining = true_ids.clone(), torch.zeros_like(remaining)
    return xt, (~remaining[0]).cpu().tolist()

print('helpers defined')

## 5 · Test 1 — are attentions actually returned?

The one that fails silently. If this raises, nothing below means anything.

In [ ]:
ids, offs = encode('The chef prepared the meal carefully.')
attentions = forward_with_attentions(ids)

assert attentions is not None and attentions[0] is not None, (
    'NO ATTENTIONS RETURNED — attn implementation is not eager'
)
assert len(attentions) == N_LAYERS, f'{len(attentions)} layers, expected {N_LAYERS}'
b, h, q, k = attentions[0].shape
assert (h, q, k) == (N_HEADS, ids.shape[1], ids.shape[1]), attentions[0].shape

rows = attentions[0][0, :, 1:, :].float().sum(-1)
print(f'PASS  {len(attentions)} layers, each {tuple(attentions[0].shape)}')
print(f'      rows sum to {rows.mean():.4f} (should be ~1.0)')
print(f'      tokens: {[tokenizer.decode([i]) for i in ids[0].tolist()]}')

## 6 · Test 2 — the BOS attention sink

Diffusion LMs park enormous mass on position 0. This is why BOS must be masked
out *before* the argmax — otherwise nearly every head trivially "predicts" BOS.

In [ ]:
bos_share, argmax_bos = [], []
for layer in range(N_LAYERS):
    a = attentions[layer][0, :, 1:, :].float()   # skip the BOS row itself
    bos_share.append(a[:, :, 0].mean().item())
    argmax_bos.append((a.argmax(-1) == 0).float().mean().item())

print(f'mean attention mass on BOS : {np.mean(bos_share):.1%}')
print(f'rows whose argmax IS BOS   : {np.mean(argmax_bos):.1%}  <- excluded before argmax\n')
for layer in range(0, N_LAYERS, 6):
    bar = '#' * int(bos_share[layer] * 50)
    print(f'  L{layer:02d} {bos_share[layer]:6.1%} {bar}')

## 7 · Test 3 — object→verb across all heads

Six sentences, each with a known object→verb dependency, scored at the final
fully-revealed frame. Does the attender's attention row land on its governing
verb? Run over all 1024 heads.

Tiny sample, so treat the numbers as a smoke signal rather than a measurement —
and note these are hand-written sentences, not UD gold.

In [ ]:
# (sentence, object noun, governing verb)
CASES = [
    ('The chef prepared the meal carefully.',      'meal',    'prepared'),
    ('She wrote a long letter yesterday.',         'letter',  'wrote'),
    ('They finally opened the heavy door.',        'door',    'opened'),
    ('The student answered the difficult question.', 'question', 'answered'),
    ('He quietly closed the wooden window.',       'window',  'closed'),
    ('The gardener planted several young trees.',  'trees',   'planted'),
]

hits = np.zeros((N_LAYERS, N_HEADS))
for text, obj, verb in CASES:
    ids, offs = encode(text)
    att = forward_with_attentions(ids)
    a_span = span_for_word(text, offs, obj)
    r_span = set(span_for_word(text, offs, verb))
    for layer in range(N_LAYERS):
        hits[layer] += np.isin(receiver_predictions(att, layer, a_span), list(r_span))

acc = hits / len(CASES)
order = np.argsort(acc, axis=None)[::-1][:10]
print(f'top heads, object -> verb  (n={len(CASES)} sentences)\n')
for flat in order:
    layer, head = np.unravel_index(flat, acc.shape)
    tag = '  <- known relational head (§4.2)' if (layer, head) == (3, 11) else ''
    print(f'  L{layer:02d} H{head:02d}  {acc[layer, head]:.2f}{tag}')

print(f'\nL3  H11 (relational, expect high) : {acc[3, 11]:.2f}')
print(f'L18 H10 (positional, expect low)  : {acc[18, 10]:.2f}')
print(f'mean over all {N_LAYERS * N_HEADS} heads          : {acc.mean():.3f}')

## 8 · Test 4 — the double dissociation, in miniature

The same two heads on an **adjacent** relation (determiner→noun) instead.
HANDOFF §4.2 predicts they invert: L18 H10 solves everything adjacent and
nothing else, L3 H11 the exact reverse. If both heads score high on both
relations, the dissociation is not reproducing.

In [ ]:
DET_CASES = [
    ('The chef prepared the meal carefully.',        'The',     'chef'),
    ('They finally opened the heavy door.',          'the',     'door'),
    ('The student answered the difficult question.', 'The',     'student'),
    ('He quietly closed the wooden window.',         'the',     'window'),
]

det_hits = np.zeros((N_LAYERS, N_HEADS))
for text, det, noun in DET_CASES:
    ids, offs = encode(text)
    att = forward_with_attentions(ids)
    a_span = span_for_word(text, offs, det)
    r_span = set(span_for_word(text, offs, noun))
    for layer in range(N_LAYERS):
        det_hits[layer] += np.isin(receiver_predictions(att, layer, a_span), list(r_span))

det_acc = det_hits / len(DET_CASES)
print(f'{"head":10s} {"object->verb":>14s} {"det->noun":>12s}   profile')
for layer, head, label in [(3, 11, 'relational'), (18, 10, 'positional')]:
    print(f'L{layer:02d} H{head:02d}   {acc[layer, head]:>14.2f} {det_acc[layer, head]:>12.2f}   {label}')

print(f'\nbest det->noun head: ', end='')
flat = int(np.argmax(det_acc))
layer, head = np.unravel_index(flat, det_acc.shape)
print(f'L{layer:02d} H{head:02d} = {det_acc[layer, head]:.2f}')

## 9 · Test 5 — intermediate denoising states

Rebuild x_t partway through denoising and confirm masking behaves: `t=0` fully
masked, `t=63` fully revealed, monotonic in between. This is what makes the
masked-state measurement possible at all.

In [ ]:
text = 'The chef prepared the meal carefully.'
true_ids, offs = encode(text)
STEPS = 64

print(f'{"t":>4} {"visible":>9}   state')
for t in [0, 8, 16, 32, 48, 63]:
    xt, visible = teacher_forced_state(true_ids, t, steps=STEPS)
    shown = ' '.join(
        tokenizer.decode([i]).strip() if v else '_'
        for i, v in zip(xt[0].tolist(), visible)
    )
    print(f'{t:>4} {sum(visible):>3}/{len(visible):<5}   {shown}')

x0, v0 = teacher_forced_state(true_ids, 0, steps=STEPS)
xf, vf = teacher_forced_state(true_ids, STEPS - 1, steps=STEPS)
assert sum(v0) == 1, 'only BOS should be visible at t=0'
assert all(vf), 'final frame must be fully revealed'
assert torch.equal(xf, true_ids), 'final frame must equal the true sentence'
print('\nPASS  t=0 fully masked, t=63 exactly the original sentence')

## 10 · Does the head still point while endpoints are masked?

The actual research question in miniature — L3 H11 at a partly-denoised frame,
with the endpoint-visibility flag printed alongside. HANDOFF §4.3 reports 7B
masked-state accuracy at only 0.108, so **do not read a low number here as a
bug**. Open question 3 also notes the metric scores against the *original*
sentence's parse, which may itself be ill-posed.

In [ ]:
LAYER, HEAD = 3, 11
text, obj, verb = CASES[0]
true_ids, offs = encode(text)
a_span = span_for_word(text, offs, obj)
r_span = set(span_for_word(text, offs, verb))

print(f'L{LAYER} H{HEAD} | "{obj}" -> "{verb}" | {text}\n')
print(f'{"t":>4} {"n_masked":>9} {"endpoints":>11} {"correct":>8}   predicted token')
for t in [0, 8, 16, 24, 32, 40, 48, 56, 63]:
    xt, visible = teacher_forced_state(true_ids, t, steps=STEPS)
    att = forward_with_attentions(xt)
    pred = receiver_predictions(att, LAYER, a_span)[HEAD]
    endpoints = list(a_span) + list(r_span)
    both_masked = not any(visible[p] for p in endpoints)
    print(f'{t:>4} {len(visible) - sum(visible):>9} '
          f'{"masked" if both_masked else "visible":>11} '
          f'{str(pred in r_span):>8}   {tokenizer.decode([xt[0, pred].item()])!r}')

---
## Summary

If cells 5–9 passed, the stack is sound: eager attention returns weights, the
sink is handled, the schedule rebuilds correctly, and the known heads behave as
HANDOFF §4.2 describes.

**What this is not.** Six hand-written sentences are not a measurement. There is
no fixed-offset null here, no held-out split, and no multiple-comparisons
diagnostic — so nothing in this notebook is quotable. Its only job is to confirm
the machinery runs before you spend GPU hours. The real run is
`colab_diffullama_gpu.ipynb`, which needs `dlmrel data` to have been run locally
first.